# Notebook 8: Plots and Graphs

**Purpose:** This notebook generates all publication-quality research plots from the saved training history.
Run this notebook **after** training is complete.

It produces the following figures:

1. **Training vs Validation Loss Curve** — Shows that the model learns and generalizes
2. **Detection Performance Curves** — Precision, Recall, F1, and mAP evolution over epochs
3. **Learning Rate Schedule** — Shows the cosine annealing warm restart pattern
4. **Precision-Recall Curve** — The primary metric for object detection research
5. **Detection Confusion Matrix** — Visual breakdown of TP, FP, FN

All plots are saved at 300 DPI and are suitable for direct inclusion in a thesis or paper.

## Step 1: Setup

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path

PROJECT_ROOT = str(Path(os.getcwd()).parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

PLOT_DIR = os.path.join(PROJECT_ROOT, 'reports', 'plots')
os.makedirs(PLOT_DIR, exist_ok=True)

# Use the paper style for all plots
plt.style.use('seaborn-v0_8-paper')
sns.set_context('paper', font_scale=1.5)

print(f"Plots will be saved to: {PLOT_DIR}")

## Step 2: Load Training History

The training notebook saves the history as a JSON file after training.
We load it here so this notebook can run independently.

In [ ]:
HISTORY_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'training_history.json')

with open(HISTORY_PATH, 'r') as f:
    history = json.load(f)

epochs = range(1, len(history['train_loss']) + 1)
print(f"History loaded. Total epochs recorded: {len(history['train_loss'])}")

## Plot 1: Training and Validation Loss Curve

This shows how the model's loss decreases over training.
A gap between train and val loss would indicate overfitting.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(epochs, history['train_loss'], label='Train Loss', linewidth=2)
plt.plot(epochs, history['val_loss'],   label='Val Loss',   linewidth=2)
plt.title('AtrionNet Hybrid: Training & Validation Convergence')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_path = os.path.join(PLOT_DIR, 'loss_curves.png')
plt.savefig(save_path, dpi=300)
plt.show()
print(f"Saved: {save_path}")

## Plot 2: Detection Performance Metrics Over Epochs

This shows how Precision, Recall, F1-Score, and mAP evolve as the model trains.

In [ ]:
plt.figure(figsize=(12, 7))
plt.plot(epochs, history['val_precision'], label='Precision', linestyle='--', linewidth=1.5)
plt.plot(epochs, history['val_recall'],    label='Recall',    linestyle='--', linewidth=1.5)
plt.plot(epochs, history['val_f1'],        label='F1 Score',  linewidth=2.5)
plt.plot(epochs, history['val_map'],       label='mAP @ 0.5', linewidth=2.5, color='gold')

plt.title('Evolution of Detection Performance on Validation Set')
plt.xlabel('Epochs')
plt.ylabel('Metric Value')
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_path = os.path.join(PLOT_DIR, 'val_metrics.png')
plt.savefig(save_path, dpi=300)
plt.show()
print(f"Saved: {save_path}")

## Plot 3: Learning Rate Schedule

Shows the cosine annealing pattern. The sharp jump is the 'warm restart' which helps escape bad minima.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(epochs, history['lr'], color='green', linewidth=2)
plt.title('Learning Rate Schedule (Cosine Annealing with Warm Restarts)')
plt.xlabel('Epochs')
plt.ylabel('Learning Rate')
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_path = os.path.join(PLOT_DIR, 'lr_schedule.png')
plt.savefig(save_path, dpi=300)
plt.show()
print(f"Saved: {save_path}")

## Plot 4: Precision-Recall Curve on Validation Set

This is the most important plot for an object detection paper.
The Area under this curve equals the Average Precision (AP).

We re-run inference on the validation set to compute the full PR curve.

In [ ]:
from src.modeling.atrion_net import AtrionNetHybrid
from src.data_pipeline.ludb_loader import LUDBLoader
from src.data_pipeline.instance_dataset import AtrionInstanceDataset
from src.engine.atrion_evaluator import compute_instance_metrics, calculate_mAP
from torch.utils.data import DataLoader

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
model = AtrionNetHybrid(in_channels=12).to(DEVICE)
model.load_state_dict(torch.load(os.path.join(PROJECT_ROOT, 'weights', 'atrion_hybrid_best.pth'), map_location=DEVICE))
model.eval()

# Load data
loader_inst   = LUDBLoader(os.path.join(PROJECT_ROOT, 'data', 'raw', 'ludb'))
signals, annotations = loader_inst.get_all_data()

val_indices   = np.load(os.path.join(PROJECT_ROOT, 'data', 'processed', 'val_indices.npy'))
val_ds        = AtrionInstanceDataset(signals[val_indices], [annotations[i] for i in val_indices], is_train=False)
val_loader_pr = DataLoader(val_ds, batch_size=1)

all_tp_lists, all_scores, total_gt = [], [], 0
with torch.no_grad():
    for i, batch in enumerate(val_loader_pr):
        sig = batch['signal'].to(DEVICE)
        out = model(sig)
        actual_idx = val_indices[i]
        targets = [{'span': (o, f)} for o, p, f in annotations[actual_idx]['p_waves']]
        res = compute_instance_metrics(out['heatmap'][0:1].cpu().numpy(), out['width'][0:1].cpu().numpy(), targets)
        all_tp_lists.append(res['tp_list'])
        all_scores.append(res['scores'])
        total_gt += res['n_gt']

m_ap, recalls, precisions = calculate_mAP(all_tp_lists, all_scores, total_gt)

plt.figure(figsize=(8, 8))
plt.step(recalls, precisions, where='post', color='b', alpha=0.8, linewidth=2, label=f'AP = {m_ap:.4f}')
plt.fill_between(recalls, precisions, step='post', alpha=0.2, color='b')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve — Instance-Level P-Wave Detection')
plt.legend(fontsize=13)
plt.xlim(0, 1)
plt.ylim(0, 1.05)
plt.grid(True)
plt.tight_layout()
save_path = os.path.join(PLOT_DIR, 'pr_curve.png')
plt.savefig(save_path, dpi=300)
plt.show()
print(f"Saved: {save_path} | AP = {m_ap:.4f}")

## Plot 5: Detection Confusion Matrix

Visualises how many P-waves the model correctly detects vs. misses or falsely reports.

In [ ]:
from src.utils.plotting import plot_confusion_matrix
from src.engine.atrion_evaluator import compute_instance_metrics

# Aggregate TP/FP/FN across the validation set
total_tp, total_fp, total_fn = 0, 0, 0

with torch.no_grad():
    for i, batch in enumerate(val_loader_pr):
        sig = batch['signal'].to(DEVICE)
        out = model(sig)
        actual_idx = val_indices[i]
        targets = [{'span': (o, f)} for o, p, f in annotations[actual_idx]['p_waves']]
        res = compute_instance_metrics(out['heatmap'][0:1].cpu().numpy(), out['width'][0:1].cpu().numpy(), targets)
        total_tp += res['tp']
        total_fp += res['fp']
        total_fn += res['fn']

cm_save_path = os.path.join(PLOT_DIR, 'confusion_matrix.png')
plot_confusion_matrix(total_tp, total_fp, total_fn, cm_save_path)

print(f"Confusion Matrix saved to: {cm_save_path}")
print(f"  TP={total_tp}, FP={total_fp}, FN={total_fn}")

## Summary: All Plots Generated

The following files have been saved to `reports/plots/`:

In [ ]:
for f in sorted(os.listdir(PLOT_DIR)):
    if f.endswith('.png'):
        print(f"  {f}")

---
**All plots generated. Your research pipeline is complete and viva-ready.**